In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F 

In [0]:
target_table=f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
results_silver_table=f"{catalog_name}.{silver_schema}.results"
sprints_silver_table=f"{catalog_name}.{silver_schema}.sprints"


In [0]:
display(results_silver_table)

In [0]:

df_results = (spark.table(results_silver_table)
              .withColumn('session_type',F.lit('RACE'))
              .drop('race_name','race_date','ingestion_timestamp','source_file')
)
            



In [0]:
df_sprints = (spark.table(sprints_silver_table)
              .withColumn('session_type',F.lit('SPRINTS'))
              .drop('race_name','race_date','ingestion_timestamp','source_file')
)

In [0]:
df_results_sprints=df_results .unionByName(df_sprints )

In [0]:
display(df_results.count()+df_sprints.count())
display(df_results_sprints.count())

In [0]:
fact_df_results_sprints=(
    df_results_sprints
    .withColumn("isWin",F.col("final_position")==1)
    .withColumn("isPodium",F.col("final_position").between(1,3))
    .withColumn("has_pints",F.col("race_points")>0)
    .drop("funal_position")
 )

In [0]:
display(fact_df_results_sprints.filter("season= 2025"))

In [0]:
(fact_df_results_sprints.
 write
 .mode("overwrite")
 .format("delta")
 .saveAsTable(target_table)

)

In [0]:
# drop_table=f"DROP TABLE IF EXISTS {target_table}"
# spark.sql(drop_table)

In [0]:
spark.table(target_table).display()